## 저장된 모델 사용하기

### 1. 라이브러리 로드

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim

import requests
from PIL import Image
from io import BytesIO
import os
import random
import shutil

# 사전학습 모델 라이브러리 추가
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# pip install icrawler
from icrawler.builtin import BingImageCrawler, BaiduImageCrawler

import matplotlib.pyplot as plt

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

### 2. 설정

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

3. ResNet18 모델구조 재생성

In [ ]:
weight = ResNet18_Weights.DEFAULT
model = resnet18(weights=weight)

model.fc = nn.Linear(
    in_features=model.fc.in_features,
    out_features=num_classes
)

model = model.to(device=device)

### 4. pth 모델파일 로드

In [ ]:
model.load_state_dict(
    torch.load(model_path, map_location=device)
)

model.eval()

### 5. 전처리 설정

In [ ]:
preprocess = weight.transforms()
preprocess

### 6. 바이두 이미지크롤링
- 테스트이미지셋 다운로드

In [ ]:
# 추가설정
testset_dir = 'dataset/test'

classes = {
    'dog photo': 'dog',
    'cat photo': 'cat',
    'bird photo': 'bird'
}

num_epoch = 5
lr = 0.001

In [ ]:
# 테스트이미지 폴더
# for keyword, class_name in classes.items():
#     save_dir = os.path.join(testset_dir, class_name)
#     os.makedirs(save_dir, exist_ok=True)

#     crawler = BaiduImageCrawler(
#         storage={'root_dir': save_dir}
#     )

#     crawler.crawl(
#         keyword=keyword,
#         max_num=150
#     )

### 7. test 데이터셋 불러오기

In [ ]:
test_dataset = ImageFolder(
    root=testset_dir,
    transform=preprocess
)
test_dataset.classes

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

### 8. test 정확도 확인

In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predict = torch.max(outputs, 1)

        correct += (predict == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total

print(f'테스트 정확도 : {test_acc:.4f}')

### 9. 실제/예측값 출력

In [ ]:
class_names = test_dataset.classes

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        prob = torch.softmax(outputs, dim=1)
        confidence, predict = torch.max(prob, 1)

        for i in range(len(labels)):
            true_label = class_names[labels[i].item()]
            pred_label = class_names[predict[i].item()]
            conf = confidence[i].item()

            print(f'실제 : {true_label} / 예측 : {pred_label} / 확률 : {conf:.4f}')

### 10. 시각화

In [ ]:
# images, labels = next(iter(test_loader))
batch_size = 8
all_batches = list(test_loader)
images, labels = all_batches[14]  

model.eval()

with torch.no_grad():
    outputs = model(images.to(device))

    prob = torch.softmax(outputs, dim=1)
    confidence, predict = torch.max(prob, 1)

plt.figure(figsize=(12, 6))

for i in range(min(8, len(images))):
    image = images[i].permute(1, 2, 0).numpy() 

    true_label = class_names[labels[i].item()]
    pred_label = class_names[predict[i].item()]

    conf = confidence[i].item()

    # 2행 4열 형태로 이미지출력
    plt.subplot(2, 4, i+ 1)
    plt.imshow(image)
    plt.title(f'True : {true_label} / Pred : {pred_label} / Acc : {conf:.2f}')
    plt.axis('off')

plt.tight_layout()
plt.show()